Testing if N is able to pass the .5 threshold using CNN, and does it improve compared to transformers. 

In [1]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import ViT
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN
from sklearn.preprocessing import LabelEncoder


In [ ]:
df_data = pd.read_csv("./data/labels_duke_N.csv")
df_data = df_data.dropna()
le = LabelEncoder()
le = le.fit(df_data["label"])
df_data["label"] = le.transform(df_data["label"])
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
    data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
)

                                               image  label
0  ./data/Duke-Breast-Cancer-MRI/Breast_MRI_001/0...    1.0
1  ./data/Duke-Breast-Cancer-MRI/Breast_MRI_002/0...    0.0
2  ./data/Duke-Breast-Cancer-MRI/Breast_MRI_003/0...    1.0
3  ./data/Duke-Breast-Cancer-MRI/Breast_MRI_004/0...    0.0
4  ./data/Duke-Breast-Cancer-MRI/Breast_MRI_005/0...    1.0


In [3]:

train_transforms = Compose([
    LoadImaged(keys="image", reader="PydicomReader"),          
    EnsureChannelFirstd(keys="image"),                      
    Orientationd(keys="image", axcodes="RAS"),              
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0),       
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                  
        keys="image", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),     
    RandRotate90d(keys="image", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="image", prob=0.2),
    EnsureTyped(keys="image"),
])

val_transforms = Compose([                                  
    LoadImaged(keys="image", reader="PydicomReader"),
    EnsureChannelFirstd(keys="image"),
    Orientationd(keys="image", axcodes="RAS"),
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99,
                                    b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),
    EnsureTyped(keys="image"),
])


/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [4]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

train_ds = CacheDataset(data=train_data, transform=train_transforms, cache_rate=1.0)
val_ds   = CacheDataset(data=val_data,   transform=val_transforms,   cache_rate=1.0)




Loading dataset: 100%|██████████| 179/179 [03:20<00:00,  1.12s/it]


In [5]:
train_loader = DataLoader(train_ds, batch_size=30, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=30, num_workers=0)

from monai.networks.nets import DenseNet121
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 4
 
model = DenseNet121(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    pretrained=False,
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)



In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(x)
        if isinstance(logits, tuple):   
            logits = logits[0]
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = total = 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["image"].to(device)
            y = batch["label"].to(device)
            logits = model(x)
            if isinstance(logits, tuple):
                logits = logits[0]
            preds = logits.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    print(f"epoch {epoch}: val acc = {accuracy_score(all_labels, all_preds)}, KWQ = {cohen_kappa_score(all_labels, all_preds, weights="quadratic")}")

epoch 0: val acc = 0.5363128491620112, KWQ = -0.016147344518733453
epoch 1: val acc = 0.5027932960893855, KWQ = -0.03906861466214462
epoch 2: val acc = 0.5810055865921788, KWQ = -0.009989825178059464
epoch 3: val acc = 0.37988826815642457, KWQ = -0.035537190082644665
epoch 4: val acc = 0.4972067039106145, KWQ = -0.03600549995203539
epoch 5: val acc = 0.39664804469273746, KWQ = 0.01104972375690616
epoch 6: val acc = 0.4748603351955307, KWQ = -0.0831026713871903
epoch 7: val acc = 0.5586592178770949, KWQ = 0.004612954186413942
epoch 8: val acc = 0.5195530726256983, KWQ = -0.02740874500193491
epoch 9: val acc = 0.5586592178770949, KWQ = 0.05732341770131821
epoch 10: val acc = 0.547486033519553, KWQ = -0.03674409914678978
epoch 11: val acc = 0.5642458100558659, KWQ = -0.005116046917893691
epoch 12: val acc = 0.3128491620111732, KWQ = -0.0966129145834802
epoch 13: val acc = 0.5251396648044693, KWQ = -0.01189083820662784
epoch 14: val acc = 0.5251396648044693, KWQ = -0.023462145110410226
epo